In [2]:
import os
import math
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


#Load dataset

In [3]:
from datasets import load_dataset

def movies_dataset():
    # 1) Cornell Movie Review (Rotten Tomatoes)
    try:
        ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")
        # Ensure a 'text' column exists
        if "text" not in ds["train"].column_names:
            for c in ds["train"].column_names:
                if any(k in c.lower() for k in ("text", "sentence", "review", "headline")):
                    ds = ds.rename_column(c, "text")
                    break
        return ds
    except Exception as e:
        print("Dataset 1 failed:", e)

    # 2) movies Phrasebank fallback
    try:
        ds = load_dataset("takala/movies_phrasebank", "sentences_allagree")
        if "sentence" in ds["train"].column_names:
            ds = ds.rename_column("sentence", "text")
        return ds
    except Exception as e:
        print("Dataset 2 failed:", e)

    # 3) Toy fallback
    from datasets import Dataset, DatasetDict
    toy = Dataset.from_dict({
        "text": [
            "A thrilling ride from start to finish.",
            "A boring, predictable mess.",
            "Brilliant performances elevate this film.",
            "The plot made no sense whatsoever.",
            "A masterpiece of modern cinema.",
        ]
    })
    return DatasetDict({"train": toy, "test": toy.select(range(2))})


# Load and inspect
ds = movies_dataset()
print(ds)
print("Columns:", ds["train"].column_names)
print("Sample:", ds["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})
Columns: ['text', 'label']
Sample: {'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1}


# **Validation**

In [5]:
from datasets import DatasetDict

# ── 1. Ensure we have a validation split ────────────────────────────────────
if "validation" not in ds:
    if "test" in ds:
        # rename existing test split → validation
        ds = DatasetDict({
            "train":      ds["train"],
            "validation": ds["test"],
        })
    else:
        # carve a validation split out of train
        split = ds["train"].train_test_split(test_size=0.05, seed=42)
        ds = DatasetDict({
            "train":      split["train"],
            "validation": split["test"],
        })

print("Splits available:", list(ds.keys()))

# ── 2. Downsample (optional) ─────────────────────────────────────────────────
max_train = 15_000
max_val   =  2_000

train_ds = ds["train"]
val_ds   = ds["validation"]

if len(train_ds) > max_train:
    train_ds = train_ds.shuffle(seed=42).select(range(max_train))
if len(val_ds) > max_val:
    val_ds = val_ds.shuffle(seed=42).select(range(max_val))

print("Train size:", len(train_ds))
print("Val size:  ", len(val_ds))

Splits available: ['train', 'validation', 'test']
Train size: 8530
Val size:   1066


# Load tokenizer + model

gpt-neo\

In [6]:
model_name = "EleutherAI/gpt-neo-125M"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# GPT-Neo may not have pad token by default
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(model_name)
base_model.resize_token_embeddings(len(tokenizer))
base_model.to(device)

print("Loaded model.")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/526M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

Loaded model.


In [7]:
# ── Accuracy check (zero-shot) ───────────────────────────────────────────────
import torch

def predict(model, text, pos=" great", neg=" terrible"):
    prompt = f"Review: {text}\nSentiment:"
    ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        logits = model(**ids).logits[0, -1]
    p = tokenizer.encode(pos, add_special_tokens=False)[0]
    n = tokenizer.encode(neg, add_special_tokens=False)[0]
    return int(logits[p] > logits[n])

def evaluate(model, dataset, n=200):
    samples = dataset.shuffle(seed=42).select(range(min(n, len(dataset))))
    correct = sum(predict(model, s["text"]) == s["label"] for s in samples)
    print(f"Accuracy: {correct}/{len(samples)} = {correct/len(samples):.2%}")

base_model.eval()
evaluate(base_model, val_ds)

Accuracy: 95/200 = 47.50%


In [8]:
lora_config = LoraConfig(
    r=8,                      # rank (common: 4, 8, 16)
    lora_alpha=16,            # scaling
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(base_model, lora_config)

def print_trainable_params(m):
    trainable = 0
    total = 0
    for _, p in m.named_parameters():
        total += p.numel()
        if p.requires_grad:
            trainable += p.numel()
    pct = 100 * trainable / total
    print(f"Total params:     {total:,}")
    print(f"Trainable params: {trainable:,}")
    print(f"Trainable %:      {pct:.3f}%")
    return pct

trainable_pct = print_trainable_params(model)
# model.to(device)

Total params:     125,493,504
Trainable params: 294,912
Trainable %:      0.235%


In [9]:
def generate(prompt, max_new_tokens=60, temperature=0.8):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.95,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

print(generate("Review: The movie was absolutely"))
print(generate("Review: This film was a complete waste of"))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Review: The movie was absolutely amazing. It really made my day. It was a great movie, and I can't wait to see what the sequel will be. This is something I've wanted to see for a long time.

It was great. The soundtrack was great. The music was great, and the sound was
Review: This film was a complete waste of time and money, but its soundtrack had the perfect balance of sound and acting. It’s just that the sound of this music is great: a sound of the world, a sound of the characters, a sound of the people, of the world. It’s also just a good


In [10]:
print(generate("Review: This film was an absolute disaster,"))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Review: This film was an absolute disaster, but I love the fact that the camera lens is on a very solid frame, so we can focus on the subject of the film. I find the screen is very clear and the picture is extremely clear; there is plenty of movement in the scene! I think I was hoping for a slightly wider frame


In [11]:
# ── Tokenize ─────────────────────────────────────────────────────────────────
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

tok_train = train_ds.map(tokenize, batched=True, remove_columns=train_ds.column_names)
tok_val   = val_ds.map(tokenize,   batched=True, remove_columns=val_ds.column_names)

tok_train.set_format("torch")
tok_val.set_format("torch")

# ── Data collator ─────────────────────────────────────────────────────────────
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # causal LM, not masked
)

print("tok_train:", len(tok_train), "tok_val:", len(tok_val))

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

tok_train: 8530 tok_val: 1066


#  

## 8) Training (3 epochs)

In [15]:
output_dir = "gptneo125m-movies-lora"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",  # ← fixed
    logging_steps=50,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,         # ← fixed
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=base_model,          # ← fixed
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    data_collator=data_collator,
)

trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss
1,4.114270,4.071691
2,4.048647,4.058028
3,4.030264,4.053476


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1602, training_loss=4.060470083381948, metrics={'train_runtime': 450.1109, 'train_samples_per_second': 56.853, 'train_steps_per_second': 3.559, 'total_flos': 1676868346183680.0, 'train_loss': 4.060470083381948, 'epoch': 3.0})

#

## 9) Evaluate: Perplexity

In [17]:
eval_metrics = trainer.evaluate()
eval_loss = eval_metrics.get("eval_loss")
ppl = math.exp(eval_loss) if eval_loss is not None else None
print("Eval loss:", eval_loss)
print("Perplexity:", ppl)

Eval loss: 4.053476333618164
Perplexity: 57.5973369783416


# Generation AFTER training

In [18]:
# ── Generation AFTER training ────────────────────────────────────────────────
print(generate("The cinematography was breathtaking but"))
print("\n---\n")
print(generate("I would not recommend this film because"))
print("\n---\n")
print(generate("The director completely failed to"))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


The cinematography was breathtaking but the final act was a little too much for the audience... that's a good thing, really.... if you look to the beginning, it will be hard to tell if the audience is more interested in the characters or if they're more interested in the plot....

---



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


I would not recommend this film because of its repetitive nature... I would also say that it has a lot of good performances... but it has a lot of bad ones... and it has just too many great ones... and that's the problem with movie reviews... in the end, the

---

The director completely failed to keep pace with the production. the actors play themselves into the viewer., and the actors have to be more believable. and a little less than... i.. m, and, of course, the performances... are too much... that is... a bit


# save lora

In [19]:
adapter_dir = "movies_lora_adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print("Saved adapter to:", adapter_dir)

Saved adapter to: movies_lora_adapter


In [22]:
from google.colab import drive
drive.mount("/content/drive")

adapter_dir = "/content/drive/MyDrive/movies_lora_adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print("Saved to Google Drive:", adapter_dir)

Mounted at /content/drive
Saved to Google Drive: /content/drive/MyDrive/movies_lora_adapter
